# Coastal Water Quality: Notebook 7, How Far Can We Trust the Maps?

> Terms are defined in [`GLOSSARY.md`](./GLOSSARY.md). This notebook follows
> [`03_water_quality.ipynb`](./03_water_quality.ipynb).

**Scene:** `20250302_030003_92_4001`, Borneo / Makassar Strait coast.

## Objective
Notebook 3 mapped turbidity, chlorophyll, and CDOM. This notebook checks where the turbidity map is reliable and where it is not. It tests four sources of uncertainty:

1. **The chosen constant `C`:** does the result depend on the saturation constant?
2. **The algorithm:** do other turbidity formulas agree?
3. **Residual glint or atmosphere:** how much would a small reflectance offset change the map?
4. **The coastline:** which pixels are too close to land or too glinty to trust?

The final confidence map marks low-confidence water and gives a clear summary of the evidence.

> **Place in the workflow.** This notebook follows Notebook 3. The same sediment signal was then converted to **absolute turbidity (FNU)** in [`04_quantitative_turbidity.ipynb`](./04_quantitative_turbidity.ipynb) and to an independent **AI sediment estimate (TSS)** in [`05_ai_water_quality.ipynb`](./05_ai_water_quality.ipynb). The broader Notebook 5 water-product comparison, evaluated across its full-water mask, gives Spearman 0.77. Notebook 6's 0.797 uses a stricter common valid-pixel same-day validation mask, so it describes a different subset. The tests here use relative red-band turbidity, but the three products share the same sediment signal, so the spatial confidence findings also apply to notebooks 4 and 5.


## 0. Setup: rebuild the water mask and Notebook 3 turbidity

We use the same memory-safe `h5py` read as before: a few bands, mask layers, the water mask, and the Notebook 3 relative turbidity index (`ρ(665)` with `C = 0.25`). SciPy supplies the rank correlation and distance-to-shore calculation used later.


In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import h5py
from scipy import ndimage
from scipy.stats import spearmanr, rankdata

def find_root(start=Path.cwd()):
    p = start.resolve()
    for cand in [p, *p.parents]:
        if (cand / "data" / "inventory").exists():
            return cand
    return p

PROJECT_ROOT = find_root()
SCENE_ID = "20250302_030003_92_4001"
DATA_DIR = PROJECT_ROOT / "data" / "coastal" / SCENE_ID
FIG_DIR  = PROJECT_ROOT / "figures"; FIG_DIR.mkdir(parents=True, exist_ok=True)
SR_PATH  = DATA_DIR / "ortho_sr_hdf5.h5"
DF, FILL = "HDFEOS/GRIDS/HYP/Data Fields", -9999.0
GSD = 33.0   # metres per pixel
assert SR_PATH.exists(), f"Missing {SR_PATH} - run: python scripts/download_coastal.py --hdf5"

with h5py.File(SR_PATH, "r") as f:
    SR = f[f"{DF}/surface_reflectance"]; WL = np.asarray(SR.attrs["wavelengths"], float)
    def band(nm):
        i = int(np.argmin(np.abs(WL - nm))); b = SR[i].astype("float32"); b[b == FILL] = np.nan
        return b
    b443, b560, b665, b861 = (band(x) for x in (443, 560, 665, 861))
    nodata = f[f"{DF}/nodata_pixels"][:]; cloud = f[f"{DF}/beta_cloud_mask"][:]; cirrus = f[f"{DF}/beta_cirrus_mask"][:]

valid = (nodata == 0) & np.isfinite(b560) & np.isfinite(b861)
clean = valid & (cloud == 0) & (cirrus == 0)
ndwi  = (b560 - b861) / (b560 + b861)
water = clean & (ndwi > 0)
wpx   = int(water.sum())
def wonly(a): return np.where(water, a, np.nan)

C0 = 0.25
turbidity = wonly(b665 / (1 - b665 / C0))

# Fixed seed makes the correlation sample reproducible.
rng = np.random.default_rng(0)
wr, wc = np.where(water)
sub = rng.choice(wr.size, size=min(8000, wr.size), replace=False)
sr_, sc_ = wr[sub], wc[sub]
print(f"water pixels: {wpx:,}   max red reflectance over water: {np.nanmax(b665[water]):.3f}")


## 1. Does the saturation constant `C` matter?

Notebook 3 used `C = 0.25` in `T = ρ/(1 - ρ/C)`. We test whether another value would change the map. For a relative map, it does not change pixel ordering while `C` stays above the brightest water reflectance. Here, `ρ(665)` reaches about 0.20, so values at or above 0.25 are monotonic stretches of the same signal. We plot the transforms and compare each map with the `C = 0.25` map using Spearman rank correlation.

**What to look for:** curves that rise without crossing and rank correlations near 1.0. A value below the brightest reflectance, such as `C = 0.15`, makes `1 - ρ/C` negative for bright water. That is why `C = 0.25` was used.


In [ ]:
Cs = [0.15, 0.25, 0.40, np.inf]
def Tform(rho, C): return rho if not np.isfinite(C) else rho / (1 - rho / C)

base_rank = (b665 / (1 - b665 / C0))[sr_, sc_]
rho_grid = np.linspace(0, 0.21, 200)

fig, ax = plt.subplots(1, 2, figsize=(14, 5))
for C in Cs:
    lab = "linear (C=inf)" if not np.isfinite(C) else f"C={C}"
    ax[0].plot(rho_grid, Tform(rho_grid, C), label=lab)
ax[0].axvline(np.nanmax(b665[water]), color="k", ls=":", lw=1, label="brightest water rho(665)")
ax[0].set_ylim(0, 1.2)        # This cap keeps the C=0.15 pole off scale.
ax[0].set_xlabel("red reflectance rho(665)"); ax[0].set_ylabel("turbidity index T")
ax[0].set_title("C>=0.25: smooth & monotonic; C=0.15 diverges at its pole (rho=0.15)"); ax[0].legend(fontsize=8)

rhos = [spearmanr(base_rank, Tform(b665, C)[sr_, sc_]).statistic for C in Cs]
ax[1].bar([str(C) if np.isfinite(C) else "linear" for C in Cs], rhos, color="steelblue")
ax[1].set_ylim(min(0.9, min(rhos) - 0.02), 1.001); ax[1].set_ylabel("Spearman rank corr vs C=0.25")
ax[1].set_title("Ranking is essentially unchanged by C")
for i, r in enumerate(rhos): ax[1].text(i, r, f"{r:.3f}", ha="center", va="bottom", fontsize=9)
fig.tight_layout(); fig.savefig(FIG_DIR / "07_c_sensitivity.png", dpi=150, bbox_inches="tight")
plt.show(); print("Spearman vs C=0.25:", {str(C): round(r, 4) for C, r in zip(Cs, rhos)})


## 2. Do different turbidity algorithms agree?

We compare three turbidity proxies: red `ρ(665)`, NIR `ρ(861)`, and the red/green ratio `ρ(665)/ρ(560)`. The red band is the default proxy for low to moderate turbidity, while the NIR band is useful at high turbidity. We compare their ranks and map the spread among their percentile ranks.

**What to look for:** red and NIR should agree strongly in the plume. Faint offshore water should show more disagreement because the signal is weak. The plume should remain visible across methods even where its edges are less certain.


In [ ]:
T_red, T_nir, T_ratio = wonly(b665), wonly(b861), wonly(b665 / b560)
r_nir   = spearmanr(T_red[sr_, sc_], T_nir[sr_, sc_]).statistic
r_ratio = spearmanr(T_red[sr_, sc_], T_ratio[sr_, sc_]).statistic

def prank(a):
    out = np.full(a.shape, np.nan); m = np.isfinite(a); out[m] = rankdata(a[m]) / m.sum(); return out
disagree = np.nanstd(np.stack([prank(T_red), prank(T_nir), prank(T_ratio)]), axis=0)

fig, ax = plt.subplots(1, 2, figsize=(14, 6))
ax[0].scatter(prank(T_red)[sr_, sc_], prank(T_nir)[sr_, sc_], s=4, alpha=0.3, label=f"NIR861 (r={r_nir:.2f})")
ax[0].scatter(prank(T_red)[sr_, sc_], prank(T_ratio)[sr_, sc_], s=4, alpha=0.3, color="C1", label=f"red/green (r={r_ratio:.2f})")
ax[0].plot([0, 1], [0, 1], "k--", lw=1); ax[0].set_xlabel("red665 rank"); ax[0].set_ylabel("other-method rank")
ax[0].set_title("Algorithm agreement (rank vs rank)"); ax[0].legend(fontsize=8)
im = ax[1].imshow(disagree, cmap="inferno", vmin=0, vmax=np.nanpercentile(disagree[water], 98))
ax[1].set_title("Where algorithms disagree (rank spread)"); ax[1].axis("off")
fig.colorbar(im, ax=ax[1], fraction=0.046)
fig.tight_layout(); fig.savefig(FIG_DIR / "07_algorithm_spread.png", dpi=150, bbox_inches="tight")
plt.show(); print(f"red vs NIR spearman={r_nir:.3f}; red vs red/green spearman={r_ratio:.3f}")


**References used here**
- Dogliotti, A.I., Ruddick, K.G., Nechad, B., Doxaran, D., & Knaeps, E. (2015). *A single algorithm to retrieve
  turbidity from remotely-sensed data in all coastal and estuarine waters.* Remote Sensing of Environment, 156,
  157-168. https://doi.org/10.1016/j.rse.2014.09.020. The paper supports the use of the red band for low to
  moderate turbidity and the NIR band for high turbidity, which is why both are compared here.


## 3. What if the atmospheric correction were slightly off?

Sun glint and imperfect atmospheric correction can add a small, roughly spectrally flat reflectance error. The black-pixel check in Notebook 2 indicates that it is small here, but not zero. We add `+0.003` to the red band, recalculate turbidity, and compare the change in bright plume water with clear offshore water.

**What to look for:** a fixed offset is a small fraction of a large plume value but a large fraction of a small clear-water value. The plume should change little, while faint offshore water should be more sensitive.


In [ ]:
delta = 0.003
relchg = wonly(np.abs((b665 + delta) - b665) / b665) * 100.0

q20, q80 = np.nanpercentile(b665[water], [20, 80])
groups = {"plume (top 20%)": water & (b665 >= q80),
          "mid":             water & (b665 > q20) & (b665 < q80),
          "clear (bot 20%)": water & (b665 <= q20)}
meds = {k: float(np.nanmedian(relchg[m])) for k, m in groups.items()}

fig, ax = plt.subplots(1, 2, figsize=(14, 6))
im = ax[0].imshow(relchg, cmap="cividis", vmin=0, vmax=np.nanpercentile(relchg[water], 95))
ax[0].set_title(f"Turbidity change from a +{delta} reflectance offset (%)"); ax[0].axis("off")
fig.colorbar(im, ax=ax[0], fraction=0.046, label="% change")
ax[1].bar(list(meds), list(meds.values()), color=["C3", "C7", "C0"])
ax[1].set_ylabel("median % change"); ax[1].set_title("Bright water is robust; faint water is fragile")
for i, v in enumerate(meds.values()): ax[1].text(i, v, f"{v:.0f}%", ha="center", va="bottom")
fig.tight_layout(); fig.savefig(FIG_DIR / "07_offset_sensitivity.png", dpi=150, bbox_inches="tight")
plt.show(); print("median % change:", {k: round(v, 1) for k, v in meds.items()})


**References used here**
- Gordon, H.R., & Wang, M. (1994). *Retrieval of water-leaving radiance and aerosol optical thickness over the
  oceans with SeaWiFS: a preliminary algorithm.* Applied Optics, 33(3), 443-452.
  https://doi.org/10.1364/AO.33.000443. This provides the NIR black-pixel atmospheric-correction framework; we
  perturb its residual to assess how a small correction error affects turbidity.
- Hedley, J.D., Harborne, A.R., & Mumby, P.J. (2005). *Simple and robust removal of sun glint for mapping
  shallow-water benthos.* International Journal of Remote Sensing, 26(10), 2107-2112.
  https://doi.org/10.1080/01431160500034086. This supports treating residual sun glint as a small, flat offset in
  water reflectance.


## 4. The coastline problem and a confidence map

Near-shore pixels have two extra risks. **Adjacency** scatters light from bright land into nearby water. **Residual glint** can raise NIR reflectance over water. We calculate distance to the nearest land pixel and flag three low-confidence groups:

- **near shore:** within about 165 m (5 pixels) of land;
- **residual glint:** NIR `ρ(861) > 0.02`;
- **noise floor:** red `ρ(665) < 0.012`, where the relative turbidity signal is mostly noise.

**What to look for:** low-confidence pixels should lie along the coast and in faint offshore water. The bright plume between them should remain high confidence.


In [ ]:
dist_px = ndimage.distance_transform_edt(water)        # Distance is measured to the nearest non-water pixel.
nearshore  = water & (dist_px < 5)                     # Five pixels is about 165 m.
glintflag  = water & (b861 > 0.02)
noisefloor = water & (b665 < 0.012)
lowconf    = water & (nearshore | glintflag | noisefloor)

pct = lambda m: 100 * m.sum() / wpx
print(f"near-shore (<165 m): {pct(nearshore):.1f}%   residual glint: {pct(glintflag):.1f}%   "
      f"noise floor: {pct(noisefloor):.1f}%   ANY low-confidence: {pct(lowconf):.1f}%")

fig, ax = plt.subplots(1, 3, figsize=(17, 5.6))
im0 = ax[0].imshow(wonly(dist_px * GSD), cmap="viridis_r", vmax=np.nanpercentile((dist_px * GSD)[water], 95))
ax[0].set_title("Distance to shore (m)"); ax[0].axis("off"); fig.colorbar(im0, ax=ax[0], fraction=0.046)

conf = np.where(water, 1.0, np.nan); conf[lowconf] = 0.0
ax[1].imshow(conf, cmap="RdYlGn", vmin=0, vmax=1)
ax[1].set_title(f"Confidence (green=trust, red=low; {pct(lowconf):.0f}% low)"); ax[1].axis("off")

hi = turbidity.copy(); hi[lowconf] = np.nan
im2 = ax[2].imshow(hi, cmap="turbo", vmin=np.nanpercentile(turbidity[water], 2), vmax=np.nanpercentile(turbidity[water], 98))
ax[2].set_title("Turbidity, high-confidence water only"); ax[2].axis("off"); fig.colorbar(im2, ax=ax[2], fraction=0.046)
fig.tight_layout(); fig.savefig(FIG_DIR / "07_confidence.png", dpi=150, bbox_inches="tight")
plt.show(); print("saved ->", Path("figures") / "07_confidence.png")


**References used here**
- Santer, R., & Schmechtig, C. (2000). *Adjacency effects on water surfaces: primary scattering approximation and
  sensitivity study.* Applied Optics, 39(3), 361-375. https://doi.org/10.1364/AO.39.000361. This supports flagging
  near-shore water because nearby bright land can affect its reflectance through atmospheric scattering.
- Hedley, J.D., Harborne, A.R., & Mumby, P.J. (2005). *Simple and robust removal of sun glint for mapping
  shallow-water benthos.* International Journal of Remote Sensing, 26(10), 2107-2112.
  https://doi.org/10.1080/01431160500034086. This supports using elevated NIR reflectance over water as a residual-glint flag.


## 5. The consolidated error budget

The four tests address different weaknesses. This final step brings them together in one view: how far each product can be trusted and why. It also uses the cube's `surface_reflectance_uncertainty` layer to estimate the red-band measurement floor over water. That is Planet's estimate of reflectance noise and sets the smallest sediment difference worth interpreting.

We use three trust levels:
- **High:** stable in the stress tests and, where possible, supported by an independent method.
- **Medium:** useful but limited by measurement noise or land-style atmospheric correction.
- **Low:** near the noise floor or confounded.

**What to look for:** plume pattern and location should score High. Absolute FNU and TSS should score Medium because their magnitude is limited by the measurement floor and atmospheric correction. Chlorophyll, CDOM, and faint offshore water should score Low. The broader Notebook 5 water-product comparison across its full-water mask, Spearman 0.77, is the key independent check. Notebook 6's 0.797 uses its stricter common valid-pixel same-day validation mask.


In [ ]:
with h5py.File(SR_PATH, "r") as f:
    SRU = f[f"{DF}/surface_reflectance_uncertainty"]
    iR = int(np.argmin(np.abs(WL - 665)))
    unc665 = SRU[iR].astype("float32"); unc665[unc665 == FILL] = np.nan
rel_floor = float(np.nanmedian((unc665 / b665)[water]) * 100)

budget = [
 ("Relative turbidity (pattern)", "High",   f"rank-stable under C (rho>={min(rhos):.2f}); red~NIR (rho={r_nir:.2f})"),
 ("Plume location",               "High",   f"offset test: plume {meds['plume (top 20%)']:.0f}% vs clear {meds['clear (bot 20%)']:.0f}% change"),
 ("Absolute turbidity FNU (4)",  "Medium", f"measurement floor ~{rel_floor:.0f}% (refl. uncertainty) + land-style AC"),
 ("AI sediment TSS (5)",         "Medium", "Notebook 5 full-water comparison: Spearman 0.77; Notebook 6 strict shared-valid same-day validation mask: 0.797"),
 ("Faint offshore water",         "Low",    "near noise floor; swings with offset & algorithm"),
 ("Chlorophyll / CDOM",           "Low",    "weak and confounded by sediment (notebook 5 concurs)"),
]
score = {"High": 3, "Medium": 2, "Low": 1}; colmap = {"High": "#2ca02c", "Medium": "#ff7f0e", "Low": "#d62728"}
fig, ax = plt.subplots(figsize=(12.5, 5))
names = [b[0] for b in budget[::-1]]
for i, (name, trust, why) in enumerate(budget[::-1]):
    ax.barh(i, score[trust], color=colmap[trust], alpha=0.85)
    ax.text(3.25, i, why, va="center", ha="left", fontsize=8)
ax.set_yticks(range(len(budget))); ax.set_yticklabels(names, fontsize=9, fontweight="bold")
ax.set_xlim(0, 9.6); ax.set_xticks([1, 2, 3]); ax.set_xticklabels(["Low", "Medium", "High"])
ax.axvline(3.1, color="0.85", lw=0.8)
ax.set_xlabel("trust level"); ax.set_title("Coastal deep-dive error budget - how far to trust each product")
fig.tight_layout(); fig.savefig(FIG_DIR / "07_error_budget.png", dpi=150, bbox_inches="tight")
plt.show(); print(f"red-band measurement floor over water (median refl. uncertainty): ~{rel_floor:.1f}%")


## Summary: what survives the stress test

| Confound | Where it matters | What we did |
|---|---|---|
| Choice of `C` | absolute scale only | showed that ranking is unchanged (Spearman about 1.0) |
| Choice of algorithm | faint offshore water | red and NIR agree strongly; high-disagreement pixels are flagged |
| Residual glint or atmosphere | faint offshore water | offset test shows a robust plume and sensitive clear water |
| Adjacency from bright land | near-shore strip | distance-to-shore buffer flags low-confidence water |
| No in-situ data | absolute units | kept the original map relative |

**The verdict:**
- The **turbidity plume is robust**. Its pattern changes little when `C`, the algorithm, or a small reflectance offset changes.
- The physics-based FNU turbidity in Notebook 4 and the AI TSS estimate in Notebook 5 agree in the broader full-water product comparison at Spearman 0.77. Notebook 6's 0.797 uses a stricter common valid-pixel same-day validation mask, so it is a different subset. This is the strongest independent check.
- **Faint offshore water** is sensitive to offsets and algorithm choice, so its exact value is not reliable.
- The **near-shore strip** is affected by adjacency and glint and is flagged.
- **Chlorophyll and CDOM** remain weak and confounded. Notebook 5 also finds low chlorophyll and CDOM that tracks sediment.

About one third of water pixels are low confidence, mainly faint offshore water and the coastal buffer. The high-confidence core is the sediment plume, supported by relative turbidity, absolute FNU, and AI TSS.

**Figures:** `07_c_sensitivity.png`, `07_algorithm_spread.png`, `07_offset_sensitivity.png`, `07_confidence.png`, and `07_error_budget.png`.

**Next:** [`coastcheck_sangatta.ipynb`](./coastcheck_sangatta.ipynb) applies this evidence to repeat-monitoring priorities.
